# Fine-Tuning IndoBERT untuk Extractive Summarization (IndoSUM)

Notebook ini melatih model IndoBERT sebagai classifier kalimat penting (extractive summarizer)
menggunakan dataset IndoSUM.

**Alur Notebook:**
- **Bagian 1–3**: Setup, Persiapan Data, Pelatihan → jalankan sekali lalu model disimpan ke Google Drive
- **Bagian 4**: Load model dari Drive → **mulai dari sini jika model sudah pernah dilatih**
- **Bagian 5**: Test model dengan URL artikel berita

In [ ]:
# SEL 1 — Instalasi pustaka

!pip uninstall -y newspaper3k lxml -q
!pip install -q transformers datasets evaluate rouge_score accelerate \
              nltk scikit-learn newspaper3k lxml_html_clean

In [ ]:
# SEL 2 — Import semua library

import os
import json
import shutil

import torch
import torch.nn.functional as F
import numpy as np
import nltk

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset
import evaluate
from sklearn.model_selection import train_test_split
from newspaper import Article
from nltk.tokenize import sent_tokenize

In [ ]:
# SEL 3 — Download resource NLTK

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

---
## Bagian 2: Persiapan Data

Upload file `train.01.jsonl` dan `dev.01.jsonl` dari dataset IndoSUM ke root Colab
sebelum menjalankan cell ini.

In [ ]:
# SEL 4 — Fungsi: buat label oracle dan load data IndoSUM

def create_extractive_labels(article, summary, threshold=0.3):
    """
    Memberi label 1 pada kalimat artikel yang cukup mirip dengan summary,
    dan label 0 pada kalimat yang tidak relevan.
    Menggunakan word-overlap sederhana sebagai sinyal oracle.
    """
    article_sentences = nltk.sent_tokenize(article)
    summary_words = set(
        word.lower()
        for sent in nltk.sent_tokenize(summary)
        for word in nltk.word_tokenize(sent)
    )

    labels = []
    for sent in article_sentences:
        sent_words = set(word.lower() for word in nltk.word_tokenize(sent))
        if not sent_words:
            labels.append(0)
            continue

        overlap = len(sent_words & summary_words)
        recall  = overlap / len(summary_words) if summary_words else 0

        # Label 1 jika overlap cukup besar (recall > threshold)
        # atau setidaknya ada 3 kata yang cocok (heuristik untuk kalimat pendek)
        labels.append(1 if recall > threshold or overlap >= 3 else 0)

    return article_sentences, labels


def load_indosum_file(file_path):
    """
    Membaca file IndoSUM (.jsonl).
    Setiap baris adalah satu artikel dengan field 'paragraphs' dan 'summary',
    keduanya berupa list-of-list-of-words.
    """
    articles, summaries = [], []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            # Corrected: Join words into sentences, then join sentences into text
            article_text = " ".join([" ".join(sentence_words) for paragraph_sentences in data['paragraphs'] for sentence_words in paragraph_sentences])
            summary_text = " ".join([" ".join(sentence_words) for sentence_words in data['summary']])
            articles.append(article_text)
            summaries.append(summary_text)
    return articles, summaries

In [ ]:
# SEL 5 — Load dan labeling data

MAX_TRAIN = 1000
MAX_VAL   = 200

train_articles, train_summaries = load_indosum_file('train.01.jsonl')
val_articles,   val_summaries   = load_indosum_file('dev.01.jsonl')

train_texts, train_labels = [], []
for art, summ in zip(train_articles[:MAX_TRAIN], train_summaries[:MAX_TRAIN]):
    sents, lbls = create_extractive_labels(art, summ)
    train_texts.extend(sents)
    train_labels.extend(lbls)

val_texts, val_labels = [], []
for art, summ in zip(val_articles[:MAX_VAL], val_summaries[:MAX_VAL]):
    sents, lbls = create_extractive_labels(art, summ)
    val_texts.extend(sents)
    val_labels.extend(lbls)

print(f"Train — Total: {len(train_texts)} | Label-0: {train_labels.count(0)} | Label-1: {train_labels.count(1)}")
print(f"Val   — Total: {len(val_texts)}   | Label-0: {val_labels.count(0)}   | Label-1: {val_labels.count(1)}")

FileNotFoundError: [Errno 2] No such file or directory: 'train.01.jsonl'

---
## Bagian 3: Tokenisasi & Pembuatan Dataset

In [ ]:
# SEL 6 — Tokenisasi dengan IndoBERT

MODEL_CHECKPOINT = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_filter(texts, labels, max_length=128):
    """Filter kalimat kosong lalu tokenisasi dalam satu batch."""
    filtered_texts, filtered_labels = zip(
        *[(t, l) for t, l in zip(texts, labels) if t.strip()]
    )
    encodings = tokenizer(
        list(filtered_texts),
        padding="max_length",
        truncation=True,
        max_length=max_length
    )
    return encodings, list(filtered_labels)

train_encodings, filtered_train_labels = tokenize_and_filter(train_texts, train_labels)
val_encodings,   filtered_val_labels   = tokenize_and_filter(val_texts,   val_labels)

print(f"Train setelah filter: {len(filtered_train_labels)} kalimat")
print(f"Val setelah filter  : {len(filtered_val_labels)} kalimat")

In [ ]:
# SEL 7 — Buat HuggingFace Dataset

class ExtractiveDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ExtractiveDataset(train_encodings, filtered_train_labels)
val_dataset   = ExtractiveDataset(val_encodings,   filtered_val_labels)

print(f"Train Dataset: {len(train_dataset)} sampel")
print(f"Val Dataset  : {len(val_dataset)} sampel")

---
## Bagian 4: Pelatihan & Simpan ke Google Drive

Jalankan sekali. Setelah selesai, model disimpan permanen ke Drive.
**Jika model sudah pernah dilatih, lewati bagian ini dan lanjut ke Bagian 5.**

In [ ]:
# SEL 8 — Mount Google Drive & definisikan path model

from google.colab import drive

drive.mount('/content/drive')

# ── PATH MODEL DI DRIVE ── ubah sesuai preferensi kamu ──
DRIVE_MODEL_PATH = "/content/drive/MyDrive/indobert_extractive_summarizer"

print(f"Model akan disimpan ke: {DRIVE_MODEL_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model akan disimpan ke: /content/drive/MyDrive/indobert_extractive_summarizer


In [ ]:
# SEL 9 — Training IndoBERT

model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2)

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./results_extractive",   # checkpoint sementara (lokal)
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Memulai fine-tuning IndoBERT...")
trainer.train()
print("Pelatihan selesai!")

In [ ]:
# SEL 10 — Simpan model & tokenizer ke Google Drive

# Hapus folder lama di Drive jika ada, agar tidak tertumpuk
if os.path.exists(DRIVE_MODEL_PATH):
    shutil.rmtree(DRIVE_MODEL_PATH)
    print(f"Folder lama dihapus: {DRIVE_MODEL_PATH}")

os.makedirs(DRIVE_MODEL_PATH, exist_ok=True)

trainer.save_model(DRIVE_MODEL_PATH)
tokenizer.save_pretrained(DRIVE_MODEL_PATH)

print(f"\nModel dan tokenizer berhasil disimpan ke:\n{DRIVE_MODEL_PATH}")
print("Isi folder:", os.listdir(DRIVE_MODEL_PATH))

---
## Bagian 5: Load Model dari Google Drive

**Mulai dari sini jika model sudah dilatih sebelumnya.**
Pastikan Drive sudah di-mount (kalau belum, jalankan cell SEL 8 terlebih dahulu).

In [ ]:
# SEL 11 — Load model & tokenizer dari Drive

# Pastikan DRIVE_MODEL_PATH sudah didefinisikan (jalankan SEL 8 jika belum)
if 'DRIVE_MODEL_PATH' not in dir():
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MODEL_PATH = "/content/drive/MyDrive/indobert_extractive_summarizer"

if not os.path.exists(DRIVE_MODEL_PATH):
    raise FileNotFoundError(
        f"Model tidak ditemukan di: {DRIVE_MODEL_PATH}\n"
        "Pastikan kamu sudah menjalankan Bagian 4 minimal sekali."
    )

print("Memuat model dari Drive...")
tokenizer = AutoTokenizer.from_pretrained(DRIVE_MODEL_PATH, local_files_only=True)
model     = AutoModelForSequenceClassification.from_pretrained(DRIVE_MODEL_PATH, local_files_only=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print(f"Model berhasil dimuat! Perangkat: {device}")
print(f"Isi folder Drive: {os.listdir(DRIVE_MODEL_PATH)}")

Memuat model dari Drive...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model berhasil dimuat! Perangkat: cpu
Isi folder Drive: ['config.json', 'model.safetensors', 'training_args.bin', 'tokenizer_config.json', 'tokenizer.json']


---
## Bagian 6: Test Model dengan URL Artikel Berita

Ganti URL di bawah dengan artikel berita berbahasa Indonesia yang ingin dirangkum.

In [ ]:
# SEL 12 — Ambil artikel dari URL

# ── Ganti URL di sini ──
URL = "https://www.cnnindonesia.com/ekonomi/20260523124540-85-1361527/listrik-padam-sumatra-pulih-bertahap-83-juta-pelanggan-sudah-menyala"

article = Article(URL, language='id')
article.download()
article.parse()

title     = article.title
text      = article.text
sentences = sent_tokenize(text)

print(f"JUDUL        : {title}")
print(f"Total Kalimat: {len(sentences)}")
print(f"\nKupasan Awal Artikel:\n{text[:500]}...")

In [ ]:
# SEL 13 — Scoring kalimat & tampilkan ringkasan

# Tokenisasi semua kalimat sekaligus (lebih efisien dari looping satu per satu)
inputs = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    probs   = F.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()

# Tentukan jumlah kalimat ringkasan (30% dari total)
compression_rate = 0.3
summary_count    = max(1, int(len(sentences) * compression_rate))

# Ambil indeks kalimat dengan skor tertinggi, urutkan sesuai posisi asli
top_indices = np.argsort(probs)[::-1][:summary_count]
top_indices = sorted(top_indices)

summary = " ".join([sentences[i] for i in top_indices])

# ── Output ──
original_words = len(text.split())
summary_words  = len(summary.split())
compression    = ((original_words - summary_words) / original_words) * 100

print(f"JUDUL: {title}\n")
print("=" * 60)
print("RINGKASAN (IndoBERT Extractive):")
print("=" * 60)
print(summary)
print("\n=== STATISTIK ===")
print(f"Kata Asli      : {original_words}")
print(f"Kata Ringkasan : {summary_words}")
print(f"Kompresi Teks  : {compression:.2f}%")

```markdown
## Bagian 7: Evaluasi dengan ROUGE Score

Untuk mengevaluasi kualitas ringkasan, kita bisa menggunakan metrik ROUGE (Recall-Oriented Understudy for Gisting Evaluation). ROUGE membandingkan ringkasan yang dihasilkan dengan ringkasan referensi (dalam kasus ini, artikel asli) berdasarkan tumpang tindih unit teks (kata, bigram, atau urutan kalimat).

Kita akan menghitung ROUGE-1 (unigram), ROUGE-2 (bigram), dan ROUGE-L (Longest Common Subsequence).
```

In [ ]:
# SEL 14 — Load 5 File JSONL Lokal (IndoSum Test Set)
import json
import os

# Daftar nama file JSONL Anda. Sesuai nama file yang Anda upload ke Colab.
# Silakan ubah list di bawah ini jika nama file Anda berbeda.
files_to_load = ['test.01.jsonl', 'test.02.jsonl', 'test.03.jsonl', 'test.04.jsonl', 'test.05.jsonl']

print("Membaca 5 file JSONL pengujian lokal...")
test_articles = []
test_summaries = []

for file_name in files_to_load:
    if os.path.exists(file_name):
        print(f"  Memuat {file_name}...")
        with open(file_name, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line)
                # Corrected: Ekstrak teks artikel (menggabungkan paragraf, kalimat, dan kata sesuai struktur IndoSum)
                article_text = " ".join([" ".join(sentence_words) for paragraph_sentences in data['paragraphs'] for sentence_words in paragraph_sentences])
                # Corrected: Ekstrak teks ringkasan referensi
                summary_text = " ".join([" ".join(sentence_words) for sentence_words in data['summary']])

                test_articles.append(article_text)
                test_summaries.append(summary_text)
    else:
        print(f"  ⚠️ File '{file_name}' tidak ditemukan. Pastikan nama filenya sama persis dengan yang di-upload.")

print(f"\n✅ Selesai! Total artikel data uji yang berhasil dimuat: {len(test_articles)}")
if test_articles:
    print("\nContoh data uji pertama:")
    print(f"  [Artikel (Kupasan)] : {test_articles[0][:120]}...")
    print(f"  [Referensi (Kunci)] : {test_summaries[0][:120]}...")

Membaca 5 file JSONL pengujian lokal...
  Memuat test.01.jsonl...
  Memuat test.02.jsonl...
  Memuat test.03.jsonl...
  Memuat test.04.jsonl...
  Memuat test.05.jsonl...

✅ Selesai! Total artikel data uji yang berhasil dimuat: 18774

Contoh data uji pertama:
  [Artikel (Kupasan)] : Jakarta , CNN Indonesia - - Dilansir AFP , seorang warga Mesir yang dipercaya sebagai wanita terberat di dunia masuk seb...
  [Referensi (Kunci)] : Eman Ahmed Abd El Aty memiliki berat badan mencapai 500 kilogram sebelum menjalankan operasi di Mumbai Maret lalu dimana...


In [ ]:
# SEL 15 — Fungsi inferensi dan evaluasi ROUGE untuk IndoSum
import numpy as np
import torch
import torch.nn.functional as F
from rouge_score import rouge_scorer as rs
from nltk.tokenize import sent_tokenize

def generate_extractive_summary(sentences, compression_rate=0.3, batch_size=64):
    """Membuat ringkasan ekstraktif dari daftar kalimat menggunakan model IndoBERT."""
    if not sentences:
        return ""

    all_probs = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            probs   = F.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.extend(probs)

    all_probs = np.array(all_probs)
    summary_count = max(1, int(len(sentences) * compression_rate))
    top_indices   = sorted(np.argsort(all_probs)[::-1][:summary_count])

    return " ".join([sentences[i] for i in top_indices])


def evaluate_on_indosum(articles, summaries, max_samples=200, compression_rate=0.3):
    """Evaluasi model pada data IndoSum lokal dan hitung rata-rata ROUGE."""
    scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    metrics = {"rouge1": [], "rouge2": [], "rougeL": []}
    total = min(max_samples, len(articles))

    print(f"Mengevaluasi {total} artikel...")
    for i in range(total):
        if (i + 1) % 50 == 0:
            print(f"  Progress: {i+1}/{total}")

        article = articles[i]
        ref_summary = summaries[i]

        sentences = sent_tokenize(article)
        if len(sentences) < 2:
            continue

        predicted = generate_extractive_summary(sentences, compression_rate)
        if not predicted:
            continue

        if ref_summary.strip():
            s = scorer.score(ref_summary, predicted)
            for key in metrics:
                metrics[key].append(s[key].fmeasure)

    results = {key: float(np.mean(vals)) if vals else 0.0 for key, vals in metrics.items()}
    return results

In [ ]:
# SEL 16 — Jalankan evaluasi

# Anda bisa menaikkan MAX_SAMPLES jika ingin menguji lebih banyak data (misal: 500 atau len(test_articles))
MAX_SAMPLES      = 200
COMPRESSION_RATE = 0.3

rouge_results = evaluate_on_indosum(
    test_articles,
    test_summaries,
    max_samples=MAX_SAMPLES,
    compression_rate=COMPRESSION_RATE
)

print("\nEvaluasi selesai!")

Mengevaluasi 200 artikel...
  Progress: 50/200
  Progress: 100/200
  Progress: 150/200
  Progress: 200/200

Evaluasi selesai!


In [ ]:
# SEL 17 — Tampilkan hasil evaluasi

def print_rouge_table(results, max_samples, compression_rate):
    line = "─" * 52
    print(f"\n{'='*52}")
    print(f"  HASIL EVALUASI ROUGE — IndoSum Lokal Test Set")
    print(f"{'='*52}")
    print(f"  Model    : IndoBERT Fine-tuned (IndoSUM)")
    print(f"  Sampel   : {max_samples} artikel")
    print(f"  Kompresi : {int(compression_rate*100)}% kalimat dipilih")
    print(f"{'='*52}\n")

    print(f"  {'Metrik':<10} {'F1-Score':>12}")
    print(f"  {line}")
    for metric_key in ["rouge1", "rouge2", "rougeL"]:
        # Mengambil nilai (val) dari dictionary results
        val = results[metric_key]
        bar_len = int(val * 40)
        bar = "█" * bar_len + "░" * (40 - bar_len)
        print(f"  {metric_key.upper():<10} {val:>8.4f}  |{bar}|")
    print(f"\n{'='*52}")

# Memanggil fungsi dengan variabel dari SEL 16
print_rouge_table(rouge_results, MAX_SAMPLES, COMPRESSION_RATE)


  HASIL EVALUASI ROUGE — IndoSum Lokal Test Set
  Model    : IndoBERT Fine-tuned (IndoSUM)
  Sampel   : 200 artikel
  Kompresi : 30% kalimat dipilih

  Metrik         F1-Score
  ────────────────────────────────────────────────────
  ROUGE1       0.4377  |█████████████████░░░░░░░░░░░░░░░░░░░░░░░|
  ROUGE2       0.3291  |█████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░|
  ROUGEL       0.3784  |███████████████░░░░░░░░░░░░░░░░░░░░░░░░░|



In [ ]:
# SEL 18 — Contoh prediksi vs referensi (5 artikel acak dari data lokal)
import random
from rouge_score import rouge_scorer as rs

scorer  = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
total_data = len(test_articles)

# Mengambil 5 sampel acak untuk inspeksi visual kualitas teks ringkasan
indices = random.sample(range(total_data), min(5, total_data))

for rank, idx in enumerate(indices, 1):
    article   = test_articles[idx]
    ref_abs   = test_summaries[idx]
    sentences = sent_tokenize(article)

    predicted = generate_extractive_summary(sentences, compression_rate=0.3)
    score_abs = scorer.score(ref_abs, predicted)

    print(f"{'='*60}")
    print(f" CONTOH {rank} (Indeks Data ke-{idx})")
    print(f"{'='*60}")
    print(f" REFERENSI  : {ref_abs[:220]}...")
    print(f"\n PREDIKSI   : {predicted[:220]}...")
    print(f"\n ROUGE-1 F1 : {score_abs['rouge1'].fmeasure:.4f}")
    print(f" ROUGE-2 F1 : {score_abs['rouge2'].fmeasure:.4f}")
    print(f" ROUGE-L F1 : {score_abs['rougeL'].fmeasure:.4f}")
    print()

 CONTOH 1 (Indeks Data ke-5435)
 REFERENSI  : Kementerian Pariwisata ( Kemenpar ) mengundang sejumlah blogger mancanegara untuk melakoni acara wisata yang bertajuk ' Trip of Wonders 2016 ' yang dimulai dari 26 September - 8 Oktober mendatang . Total terdapat 25 blog...

 PREDIKSI   : Jakarta , CNN Indonesia - - Di zaman yang serba canggih ini , tidak sulit untuk mendapatkan informasi sebelum mengunjungi suatu tempat wisata . Melihat antusiasme yang sangat besar , akhirnya Kementerian Pariwisata ( Kem...

 ROUGE-1 F1 : 0.3139
 ROUGE-2 F1 : 0.1810
 ROUGE-L F1 : 0.2152

 CONTOH 2 (Indeks Data ke-334)
 REFERENSI  : Tema psikosis – atau gangguan kejiwaan – sering diadopsi di permainan - permainan video bertema horor , salah satunya yaitu Hellblade . Audio dan voice acting merupakan bagian terbaik di Hellblade . Hal unik lain yaitu s...

 PREDIKSI   : Call of Cthulhu , Amnesia , Eternal Darkness merupakan beberapa judul yang memanfaat - kannya . Dideskripsikan sebagai ‘ game independen AAA’ 